# Exploratory Analysis — Telemetry Dataset

Loads `datasets/sample_telemetry.csv` (schema: `datasets/schema.py`) and sanity-checks it before feeding `ai-engine/anomaly_detection` and `ai-engine/prediction`.

In [ ]:
import pandas as pd

df = pd.read_csv("../datasets/sample_telemetry.csv", parse_dates=["timestamp"])
df.head()

In [ ]:
df.describe()

In [ ]:
df["failed"].value_counts(normalize=True)

## Fit the anomaly detector on the healthy rows

In [ ]:
import sys
sys.path.insert(0, "../ai-engine")

from anomaly_detection.isolation_forest import AnomalyDetector

healthy = df[~df["failed"]]
detector = AnomalyDetector(contamination=0.05).fit(healthy)
scored = detector.score(df)
scored[["pod", "failed", "anomaly_score", "is_anomaly"]].sort_values("anomaly_score", ascending=False).head(10)

## Train the failure predictor

In [ ]:
from prediction.failure_predictor import FailurePredictor

predictor = FailurePredictor().fit(df)
sample = df.sample(5, random_state=0)
for row, result in zip(sample.to_dict("records"), predictor.predict(sample)):
    print(row["pod"], "->", result)